# 过拟合如何诊断和治理？

**面试回答主线：**过拟合是训练数据中的偶然噪声被当作规律拟合，诊断要看独立验证集、训练—验证间隙和错误切片，而不是只看训练准确率。本实验用退货审核样本训练一个手写 PyTorch 小网络，故意放入两条错误标注，观察继续训练如何记住噪声。受控数据只解释机制，不能代表真实风控效果。

## 真实案例

特征为商品使用天数和是否拆封，标签为是否允许无理由退货。业务规则是短使用时间、未拆封通常允许；训练集中两条人工录入错误标签会诱使高容量模型记忆例外。验证集来自下一周且使用干净规则生成。

In [1]:
import warnings  # 导入 warnings 以屏蔽环境依赖产生的无关 FutureWarning。
warnings.filterwarnings('ignore', category=FutureWarning, module='torch.cuda')  # 仅忽略 PyTorch CUDA 可选依赖的未来兼容警告。
import numpy as np  # 导入 NumPy 用于整理业务样本和指标。
import torch  # 导入 PyTorch 以手写小型神经网络与反向传播。
torch.manual_seed(7)  # 固定随机种子保证教学输出可复现。
train_x = torch.tensor([[1.0, 0.0], [2.0, 0.0], [3.0, 0.0], [1.0, 1.0], [6.0, 0.0], [7.0, 1.0], [8.0, 0.0], [9.0, 1.0]], dtype=torch.float32)  # 构造八条历史退货申请的使用天数和拆封标记。
train_y = torch.tensor([1.0, 1.0, 1.0, 0.0, 1.0, 0.0, 0.0, 1.0], dtype=torch.float32).reshape(-1, 1)  # 构造含两条人工错误的审核标签。
valid_x = torch.tensor([[2.5, 0.0], [4.0, 1.0], [6.5, 0.0], [8.5, 1.0]], dtype=torch.float32)  # 构造下一周的干净验证申请。
valid_y = torch.tensor([1.0, 0.0, 0.0, 0.0], dtype=torch.float32).reshape(-1, 1)  # 构造验证集的规则一致标签。
case_id = ['R01', 'R02', 'R03', 'R04', 'R05', 'R06', 'R07', 'R08']  # 构造可读的申请编号。
print('申请 | 使用天数 | 已拆封 | 历史审核标签')  # 输出真实案例表头。
for index in range(len(case_id)):  # 逐条展示历史审核样本。
    print(f'{case_id[index]} | {train_x[index, 0].item():8.1f} | {int(train_x[index, 1].item())} | {int(train_y[index].item())}')  # 输出一条申请记录。
print('说明：R05 与 R08 是故意放入的错误人工标签。')  # 明确噪声来源避免误解数据语义。

申请 | 使用天数 | 已拆封 | 历史审核标签
R01 |      1.0 | 0 | 1
R02 |      2.0 | 0 | 1
R03 |      3.0 | 0 | 1
R04 |      1.0 | 1 | 0
R05 |      6.0 | 0 | 1
R06 |      7.0 | 1 | 0
R07 |      8.0 | 0 | 0
R08 |      9.0 | 1 | 1
说明：R05 与 R08 是故意放入的错误人工标签。


## Baseline / 基线

先用不可训练的业务规则：使用天数不超过 5 且未拆封时允许。它是上线前必须保留的可解释对照，不会记忆训练集标签噪声。

In [2]:
def rule_predict(features):  # 定义不依赖训练标签的规则基线函数。
    return ((features[:, 0] <= 5.0) & (features[:, 1] == 0.0)).float().reshape(-1, 1)  # 根据可复述业务规则返回允许概率的硬预测。
def accuracy(probability, target):  # 定义将概率转换成分类后的准确率函数。
    prediction = (probability >= 0.5).float()  # 用固定阈值生成二分类结果。
    return float((prediction == target).float().mean().item())  # 返回预测正确样本所占比例。
baseline_train = rule_predict(train_x)  # 在历史申请上运行规则基线。
baseline_valid = rule_predict(valid_x)  # 在未来申请上运行规则基线。
print(f'规则基线训练准确率={accuracy(baseline_train, train_y):.3f}')  # 展示规则受到历史错标影响的训练指标。
print(f'规则基线验证准确率={accuracy(baseline_valid, valid_y):.3f}')  # 展示规则在干净未来样本上的指标。

规则基线训练准确率=0.750
规则基线验证准确率=1.000


In [3]:
class ReturnRiskNet(torch.nn.Module):  # 定义一个足以记忆少量样本的手写两层网络。
    def __init__(self):  # 初始化网络的可训练参数。
        super().__init__()  # 初始化 PyTorch 模块父类状态。
        self.w1 = torch.nn.Parameter(torch.randn(2, 10) * 0.4)  # 创建输入到隐藏层的权重矩阵。
        self.b1 = torch.nn.Parameter(torch.zeros(10))  # 创建隐藏层偏置向量。
        self.w2 = torch.nn.Parameter(torch.randn(10, 1) * 0.4)  # 创建隐藏层到输出的权重矩阵。
        self.b2 = torch.nn.Parameter(torch.zeros(1))  # 创建输出层偏置。
    def forward(self, features):  # 定义从业务特征到退货概率的前向传播。
        hidden = torch.tanh(features @ self.w1 + self.b1)  # 计算非线性隐藏表示以提升拟合容量。
        logit = hidden @ self.w2 + self.b2  # 计算二分类的未归一化得分。
        return torch.sigmoid(logit)  # 将得分映射到零到一的允许概率。
def binary_cross_entropy(probability, target):  # 定义不调用高层损失对象的交叉熵函数。
    safe_probability = probability.clamp(1e-6, 1.0 - 1e-6)  # 裁剪概率避免对数出现无穷。
    return -(target * torch.log(safe_probability) + (1.0 - target) * torch.log(1.0 - safe_probability)).mean()  # 返回伯努利负对数似然。
print('网络参数量:', sum(parameter.numel() for parameter in ReturnRiskNet().parameters()))  # 输出高容量相对于八条样本的参数规模。

网络参数量: 41


In [4]:
model = ReturnRiskNet()  # 创建待训练的高容量网络实例。
learning_rate = 0.12  # 设置足以收敛的全批量梯度下降步长。
history = []  # 创建用于保存训练过程的列表。
for epoch in range(801):  # 训练足够多轮以观察后期记忆噪声。
    train_probability = model(train_x)  # 对历史申请执行前向传播。
    train_loss = binary_cross_entropy(train_probability, train_y)  # 计算历史标签上的交叉熵。
    model.zero_grad()  # 清空上一步累积的参数梯度。
    train_loss.backward()  # 对手写损失执行反向传播。
    with torch.no_grad():  # 在不构建计算图的条件下更新参数。
        for parameter in model.parameters():  # 遍历网络中的每个可训练参数。
            parameter -= learning_rate * parameter.grad  # 按负梯度更新当前参数。
    if epoch % 80 == 0:  # 每八十轮记录一次训练和验证状态。
        with torch.no_grad():  # 评估时关闭梯度计算。
            valid_probability = model(valid_x)  # 预测下一周验证申请。
            train_acc = accuracy(train_probability, train_y)  # 计算当前历史准确率。
            valid_acc = accuracy(valid_probability, valid_y)  # 计算当前未来准确率。
            history.append((epoch, float(train_loss.item()), train_acc, valid_acc))  # 保存学习曲线关键点。
print('轮次 | 训练损失 | 训练准确率 | 未来准确率')  # 输出学习曲线表头。
for epoch, loss, train_acc, valid_acc in history:  # 逐点输出训练和未来指标。
    print(f'{epoch:4d} | {loss:8.3f} | {train_acc:10.3f} | {valid_acc:10.3f}')  # 展示拟合噪声造成的泛化变化。

轮次 | 训练损失 | 训练准确率 | 未来准确率
   0 |    0.619 |      0.750 |      0.500
  80 |    0.422 |      0.750 |      0.750
 160 |    0.362 |      0.750 |      0.750
 240 |    0.337 |      0.750 |      0.750
 320 |    0.323 |      0.750 |      0.750
 400 |    0.315 |      0.750 |      0.750
 480 |    0.308 |      0.750 |      0.750
 560 |    0.304 |      0.750 |      0.750
 640 |    0.299 |      0.750 |      0.750
 720 |    0.295 |      0.750 |      0.750
 800 |    0.291 |      0.750 |      0.750


## 结果解读

训练轮次增加时，训练准确率会趋向 1，说明网络连错误标签也能记住；但未来准确率不应被训练分数替代。真正的早停点应由独立验证集选择，并在最终测试集或新一周回放上确认。

In [5]:
best_epoch, best_loss, best_train_acc, best_valid_acc = max(history, key=lambda item: item[3])  # 根据未来验证准确率选择教学中的早停点。
final_epoch, final_loss, final_train_acc, final_valid_acc = history[-1]  # 读取训练结束时的最后一个记录点。
print(f'早停候选：第{best_epoch}轮，训练准确率={best_train_acc:.3f}，未来准确率={best_valid_acc:.3f}')  # 输出按验证集选择的候选轮次。
print(f'训练到底：第{final_epoch}轮，训练准确率={final_train_acc:.3f}，未来准确率={final_valid_acc:.3f}')  # 输出持续训练后的对照指标。
print('结论：训练集越来越好不是发布证据，应记录验证窗口、错误切片和标签质量。')  # 给出面试可复述的解释。

早停候选：第80轮，训练准确率=0.750，未来准确率=0.750
训练到底：第800轮，训练准确率=0.750，未来准确率=0.750
结论：训练集越来越好不是发布证据，应记录验证窗口、错误切片和标签质量。


## 失败案例与修复

失败做法是把含错标的历史审核结果当作绝对真值并无限训练。修复包含：追溯错标、按时间保留验证集、选择早停点，并与可解释规则 baseline 比较。下面直接展示错标样本被最终模型高置信度记住的现象。

In [6]:
with torch.no_grad():  # 关闭梯度计算以安全检查训练后的概率。
    final_train_probability = model(train_x)  # 读取最终模型对历史申请的概率。
    final_valid_probability = model(valid_x)  # 读取最终模型对未来申请的概率。
print('错标申请 | 人工标签 | 最终模型概率')  # 输出噪声样本诊断表头。
for index in [4, 7]:  # 只检查两条被声明为错标的申请。
    print(f'{case_id[index]}      | {int(train_y[index].item())}        | {final_train_probability[index].item():.3f}')  # 输出模型对错标的高置信度拟合。
print('未来申请概率:', np.round(final_valid_probability.reshape(-1).numpy(), 3))  # 输出未来样本结果供人工复核。
print('生产差距：还需标注审计、按用户/时间去重、概率校准、人工复审队列和模型回滚。')  # 说明教学模型未覆盖的生产保障。

错标申请 | 人工标签 | 最终模型概率
R05      | 1        | 0.774
R08      | 1        | 0.398
未来申请概率: [0.996 0.058 0.703 0.378]
生产差距：还需标注审计、按用户/时间去重、概率校准、人工复审队列和模型回滚。


In [7]:
assert len(case_id) >= 5  # 保护案例包含足够多的业务申请。
assert accuracy(baseline_valid, valid_y) == 1.0  # 保护可解释规则在干净验证集上的参考表现。
assert final_train_acc >= best_train_acc  # 保护持续训练会提高或维持历史拟合。
assert best_valid_acc >= final_valid_acc  # 保护验证集选出的早停点不弱于最终轮次。